In [13]:
# 0myStrategy/Long Call.ipynb

import os
import sys
from pathlib import Path

notebook_path = os.path.abspath('')  # مسیر فعلی
root_dir = Path(notebook_path).parent
sys.path.append(str(root_dir))

import pandas as pd

from data.cleaner import DataCleaner
from data.downloader import MarketDownloader

df_raw =  MarketDownloader.from_tsetmc_direct()
df_cleaned = DataCleaner.clean(df_raw)
df_final = DataCleaner.add_derived_columns(df_cleaned)

# تغيير قيمت نماد پايه برای نمادی که قبلا داخل ‍رتفو دارم و خرید
# UnderlyingTicker = 'خودرو'
# if UnderlyingTicker in df_final['UnderlyingTicker'].values:
#     mask_self = df_final['UnderlyingTicker'] == UnderlyingTicker
#     df_final.loc[mask_self, 'UnderlyingPrice'] = 528
# Ticker = 'ضفزر508'
# df_final = df_final[df_final['Ticker'] == Ticker].reset_index(drop=True)
# df_final['AskPrice'] = 27600

df_final.head(2)


,Ticker,Name,StrikePrice,UnderlyingTicker,UnderlyingPrice,MaturityDate,DaysToMaturity,OpenPositions,Volume,Value,...,AskVolume,InstrumentCode,InstrumentCode-UA,IntrinsicValue,MidPrice,TimeValue,Moneyness,OptionStatus,SpreadPct,PremiumOverIntrinsic
0,ضهرم5025,اختيارخ اهرم-22000-1405/05/28,22000,اهرم,52097,1405-05-28,6,17282,739,3.816713e+10,...,50,62171799977328589,17914401175772326,30097.0,27500.0,0.0,2.368045,ITM,0.218036,1.013324
1,ضهرم5026,اختيارخ اهرم-24000-1405/05/28,24000,اهرم,52097,1405-05-28,6,3349,142,7.333874e+09,...,10,23368996807956154,17914401175772326,28097.0,13514.5,0.0,2.170708,ITM,1.995560,0.934833


In [14]:
def long_call_with_fees(premium_call, stock_price, strike_price, contract_size,
                           opt_buy_commission, exercise_fee_rate, days):

    # ========== 1. محاسبه هزینه‌های ورود ==========
    premium_total = -round(premium_call * contract_size, 0)
    entry_fee = round(premium_total * opt_buy_commission, 0)
    
    # ========== 2. سرمایه اولیه (خروج نقدی) ==========
    initial_investment = premium_total + entry_fee  # منفی چون پول پرداخت می‌کنیم.
    
    # ========== 3. محاسبه سود ناخالص در سررسید ==========
    # اگر قیمت پایه بالاتر از قیمت اعمال باشد، سود داریم
    intrinsic_value = max(0, stock_price - strike_price) * contract_size
    
    # ========== 4. کارمزد اعمال (فقط در صورت سوددهی) ==========
    exercise_fee = 0
    if stock_price > strike_price:
        settlement_amount = strike_price * contract_size
        exercise_fee = -round(settlement_amount * exercise_fee_rate, 0)
        
    # ========== 5. سود خالص نهایی ==========
    # سود خالص = ارزش ذاتی - حق‌الزام پرداختی - کارمزدها
    net_profit = intrinsic_value + initial_investment + exercise_fee
    
    # ========== 6. بازده درصدی ==========
    profit_percent = round((net_profit / abs(initial_investment)) * 100, 2) if initial_investment != 0 else 0
    monthly_return = round(profit_percent * (30 / days), 2)
    
    # ========== 7. نقطه سربه‌سر (قیمت پایه در سررسید) ==========
    # باید هزینه کارمزدها را نیز به حق‌الزام اضافه کنیم
    total_cost_per_share = premium_call + (abs(entry_fee) / contract_size) + (abs(exercise_fee) / contract_size) if stock_price > strike_price else premium_call
    break_even_price = round(strike_price + total_cost_per_share, 0)
    
    # ========== 7.1. درصد قیمت پایه فعلی نسبت به نقطه سربه‌سر ==========
    # نشان می‌دهد قیمت پایه فعلی چند درصد با نقطه سربه‌سر فاصله دارد
    if break_even_price != 0:
        # اگر break_even_price > stock_price: یعنی باید قیمت بالا برود (درصد مثبت)
        # اگر break_even_price < stock_price: یعنی قیمت پایین‌تر از نقطه سربه‌سر است (درصد منفی)
        break_even_percent = round(((break_even_price - stock_price) / stock_price) * 100, 2)
    else:
        break_even_percent = 0
    

    return {
        'net_profit': net_profit,
        'profit_percent': profit_percent,
        'monthly_return': monthly_return,
        'break_even_price': break_even_price,
        'break_even_percent': break_even_percent,
        'intrinsic_value': intrinsic_value,
        'fees_total': entry_fee + exercise_fee}

In [15]:
filter_option = df_final[
    (df_final['DaysToMaturity'] > 2.0) &
    (df_final['Type'].apply(lambda x: x.name == 'CALL'))].copy()

EXCLUDED_UNDERLYING = ['اهرم']
EXCLUDED_NAME_PATTERN = ['1405/04', '1405-04']
exclude_mask = (
    (filter_option['UnderlyingTicker'].isin(EXCLUDED_UNDERLYING)) & 
    (filter_option['Name'].str.contains('|'.join(EXCLUDED_NAME_PATTERN), na=False)))

filter_option = filter_option[~exclude_mask].copy()

# filter_option = filter_option[filter_option['UnderlyingTicker'].isin(EXCLUDED_UNDERLYING)]

from config import (
    get_commission_rate,
    get_exercise_fee_rate,
    get_symbol_kind,
    get_symbol_market,
)

results_fee = []
for underlying_symbol, group in filter_option.groupby('UnderlyingTicker'):
    market = get_symbol_market(underlying_symbol)
    kind = get_symbol_kind(underlying_symbol)

    opt_buy_commission = get_commission_rate(market, 'option', True)
    exercise_fee_rate = get_exercise_fee_rate(market, kind)

    for index, item in group.iterrows():
        # استخراج اطلاعات مورد نیاز
        ticker = item['Ticker']
        strike_price = item['StrikePrice']
        premium_call = item['AskPrice']
        stock_price = item['UnderlyingPrice']
        contract_size = item['ContractSize']
        days = item['DaysToMaturity']
        
        # محاسبات با کارمزد
        results = long_call_with_fees(
            premium_call, stock_price, strike_price, contract_size,
            opt_buy_commission, exercise_fee_rate, days)

        # ذخیره نتایج در دیکشنری
        results_fee.append({
            'underlying': underlying_symbol,
            'stock_price': stock_price,
            'option_symbol': ticker,
            'strike': strike_price,
            'premium': round(premium_call, 0),
            'stock_price': round(stock_price, 0),
            'net_profit': results['net_profit'],
            'profit_percent': results['profit_percent'],
            'monthly_return_%': results['monthly_return'],
            'break_even_price': results['break_even_price'],
            'break_even_percent': results['break_even_percent'],
            'days_to_maturity': days,
            'volume': int(item.get('Volume', 0))})

result_df_fee = pd.DataFrame(results_fee)
result_df_fee = result_df_fee[result_df_fee['break_even_percent'] <= 12]
result_df_fee = result_df_fee.sort_values('break_even_percent', ascending=True).reset_index(drop=True)
# result_df_fee = result_df_fee.sort_values(
#     ['break_even_percent', 'monthly_return_%'], ascending=[True, True]).reset_index(drop=True)

In [16]:
from datetime import datetime
import pandas as pd
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

header_font = Font(name='Segoe UI', size=11, bold=True, color='FFFFFF')
header_fill = PatternFill(start_color='1F4E78', end_color='1F4E78', fill_type='solid')
alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
body_font = Font(name='Segoe UI', size=10)
gray_font = Font(color='808080', italic=True, name='Segoe UI', size=10)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"result_long_call{timestamp}.xlsx"
filename = f"result_long_call.xlsx"

with pd.ExcelWriter(filename, engine='openpyxl') as writer:
    result_df_fee.to_excel(writer, sheet_name='long_call', index=False)
    worksheet = writer.sheets['long_call']
    # اعمال استایل به هدر اصلی
    for col_idx in range(1, len(result_df_fee.columns) + 1):
        cell = worksheet.cell(row=1, column=col_idx)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = alignment

    # تزریق هوشمند استایل بدنه کدهای مالی
    columns_list = result_df_fee.columns.tolist()
    for row_idx, row in enumerate(result_df_fee.itertuples(index=False), start=2):
        for col_idx, col_name in enumerate(columns_list, start=1):
            cell = worksheet.cell(row=row_idx, column=col_idx)
            val = row[col_idx - 1]

            if val is None or pd.isna(val):
                cell.value = "-"
                cell.font = gray_font
                cell.alignment = alignment
                continue

            cell.alignment = alignment        
            cell.font = body_font

    worksheet.auto_filter.ref = f"A1:{get_column_letter(len(result_df_fee.columns))}{len(result_df_fee) + 1}"
    worksheet.freeze_panes = 'A2'
    # تنظیم خودکار و دقیق عرض ستون‌ها بر اساس طول کاراکترهای فارسی و انگلیسی
    for col in worksheet.columns:
        max_len = 0
        for cell in col:
            val = str(cell.value or '')
            actual_len = sum(2 if ord(c) > 128 else 1 for c in val)
            if actual_len > max_len:
                max_len = actual_len
        col_letter = get_column_letter(col[0].column)
        worksheet.column_dimensions[col_letter].width = min((max_len + 4), 50)
